# 03 — Confidence Interval & Credible Interval

**Research Questions yang dibahas:**
- RQ1: Berapa probabilitas sebuah PR di pandas di-merge, dan seberapa yakin kita dengan estimasi tersebut? (CI Bernoulli + Credible Interval)
- RQ2: Berapa interval kepercayaan untuk rata-rata jumlah issue per minggu sebelum dan sesudah pandas 2.0? (CI Poisson)

**Member:** bintang firaas ramadhan — Inference Analyst
**Tujuan notebook ini:** Menghitung dan menginterpretasikan confidence interval (frequentist) dan credible interval (Bayesian) berdasarkan hasil estimasi MLE & Beta posterior dari Member B (notebook `02_estimation.ipynb`).

## AI Usage Disclosure

**Member:** bintang firaas ramadhan — Inference Analyst | **Tools used:** Claude

| Task | Tool | Prompt summary | Output modified? |
| ---- | ---- | -------------- | ----------------- |
| Menulis fungsi `confidence_interval`, `ci_bernoulli`, `ci_poisson`, `credible_interval` di `src/inference.py` | Claude | "Implementasikan fungsi CI dan credible interval sesuai rumus Tsun (2020) p.300 dan p.269" | [isi: Ya/Tidak — jelaskan jika ada perubahan] |
| Menyusun kerangka/struktur notebook ini (urutan sel, judul bagian) | Claude | "Bantu susun struktur notebook 03 yang konsisten dengan notebook 02" | [isi: Ya/Tidak] |

**Written entirely without AI:** Semua cell interpretasi (penjelasan makna CI dan credible interval untuk proyek pandas), dan kesimpulan pada bagian Ringkasan & Handoff.

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta as beta_dist
from inference import confidence_interval, ci_bernoulli, ci_poisson, credible_interval

plt.rcParams['figure.figsize']    = (12, 5)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

# Load hasil estimasi dari Member B (handoff dari notebook 02)
est = pd.read_csv('../data/clean/estimation_summary.csv', index_col=0)['nilai']

print('=== Hasil Estimasi dari Member B ===')
print(est)

---
# BAGIAN 1 — CONFIDENCE INTERVAL UNTUK PR MERGE RATE (RQ1)
### Pertanyaan: Seberapa yakin kita dengan estimasi θ̂ = k/n untuk probabilitas PR di-merge?

**Formula CI (normal approximation, Wald interval)** (Tsun, 2020, p. 300):
$$\hat{\theta} \pm z \cdot \sqrt{\frac{\hat{\theta}(1-\hat{\theta})}{n}}$$

di mana $\hat{\theta} = k/n$ adalah hasil MLE Bernoulli dari Member B (notebook 02).

In [ ]:
k = int(est['k_merged'])
n = int(est['n_prs'])

ci_bern = ci_bernoulli(k=k, n=n, confidence=0.95)

print('=== 95% Confidence Interval — PR Merge Rate ===')
print(f"  theta_hat     : {ci_bern['theta_hat']:.4f}")
print(f"  k (merged)    : {ci_bern['k']:,}")
print(f"  n (total PR)  : {ci_bern['n']:,}")
print(f"  z             : {ci_bern['z']:.4f}")
print(f"  Margin error  : {ci_bern['margin_of_error']:.4f}")
print(f"  95% CI        : ({ci_bern['lower']:.4f}, {ci_bern['upper']:.4f})")

**Interpretasi:** [TUGAS KAMU — tulis interpretasi CI di sini]

Panduan (hapus setelah ditulis ulang dengan kalimatmu sendiri):
- Sebutkan nilai interval (lower, upper) dan apa artinya dalam konteks merge rate PR pandas.
- Gunakan bahasa **frekuentis yang benar**. JANGAN menulis "ada probabilitas 95% bahwa θ berada di interval ini" — itu salah secara konsep frequentist.
- Kalimat yang benar kira-kira berbentuk: "Jika prosedur pengambilan sampel dan penghitungan interval ini diulang berkali-kali, sekitar 95% dari interval yang dihasilkan akan memuat nilai θ yang sebenarnya."
- Bandingkan lebar interval ini dengan besarnya n — apa yang bisa disimpulkan tentang presisi estimasi?

## 1.1 Credible Interval — Pendekatan Bayesian

**Formula Credible Interval** (Tsun, 2020, p. 269):

Interval $[lower, upper]$ sedemikian sehingga $P(lower \le \theta \le upper \mid \text{data}) = 0.95$, dihitung dari kuantil distribusi Beta($\alpha$, $\beta$) posterior hasil Member B.

In [ ]:
alpha = est['beta_alpha']
beta_param = est['beta_beta']

cri = credible_interval(alpha=alpha, beta=beta_param, confidence=0.95)

print('=== 95% Credible Interval — PR Merge Rate (Bayesian) ===')
print(f"  alpha = {cri['alpha']}, beta = {cri['beta']}")
print(f"  95% Credible Interval : ({cri['lower']:.4f}, {cri['upper']:.4f})")

print('\n=== Perbandingan ===')
print(f"  95% CI (frequentist)        : ({ci_bern['lower']:.4f}, {ci_bern['upper']:.4f})")
print(f"  95% Credible Interval (Bayes): ({cri['lower']:.4f}, {cri['upper']:.4f})")

In [ ]:
theta_vals = np.linspace(0.55, 0.78, 500)
pdf_vals   = beta_dist.pdf(theta_vals, alpha, beta_param)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(theta_vals, pdf_vals, color='steelblue', linewidth=2, label='Beta posterior')
ax.fill_between(theta_vals, pdf_vals, alpha=0.15, color='steelblue')

ax.axvline(ci_bern['lower'], color='red', linestyle='--', label='Batas CI 95% (frequentist)')
ax.axvline(ci_bern['upper'], color='red', linestyle='--')

ax.axvline(cri['lower'], color='green', linestyle=':', label='Batas Credible Interval 95% (Bayes)')
ax.axvline(cri['upper'], color='green', linestyle=':')

ax.set_title('Confidence Interval vs Credible Interval — PR Merge Rate')
ax.set_xlabel(r'$\theta$ (probabilitas merge)')
ax.set_ylabel('Densitas')
ax.legend()
plt.tight_layout()
plt.savefig('../data/clean/plot_ci_vs_credible.png', dpi=150)
plt.show()

**Interpretasi:** [TUGAS KAMU — tulis interpretasi di sini]

Panduan (hapus setelah ditulis ulang dengan kalimatmu sendiri):
- Apakah kedua interval (CI frequentist dan credible interval Bayesian) hampir berimpit? Mengapa hal itu masuk akal mengingat n = 3.000 dan prior yang digunakan adalah uniform?
- Untuk credible interval, bahasa probabilitas LANGSUNG terhadap θ memang valid secara Bayesian (berbeda dengan CI frequentist) — jelaskan kontras ini secara singkat.

---
# BAGIAN 2 — CONFIDENCE INTERVAL UNTUK RATA-RATA ISSUE PER MINGGU (RQ2)
### Pertanyaan: Berapa interval kepercayaan untuk λ (rata-rata issue/minggu) sebelum dan sesudah pandas 2.0?

**Formula CI Poisson (normal approximation)** (Tsun, 2020, p. 300):
$$\hat{\lambda} \pm z \cdot \sqrt{\frac{\hat{\lambda}}{n}}$$

di mana $\hat{\lambda}$ adalah hasil MLE Poisson dari Member B (notebook 02), dan $\sigma = \sqrt{\hat{\lambda}}$ karena pada distribusi Poisson, $\text{Var}(X) = E[X] = \lambda$.

In [ ]:
# Hitung ulang CI Poisson menggunakan data mingguan asli (bukan hanya lambda_hat)
df_issues = pd.read_csv('../data/clean/issues_clean.csv', parse_dates=['created_at'])

weekly_pre  = df_issues[df_issues['post_v2'] == 0].groupby('week').size().values
weekly_post = df_issues[df_issues['post_v2'] == 1].groupby('week').size().values

ci_pois_pre  = ci_poisson(weekly_pre,  confidence=0.95)
ci_pois_post = ci_poisson(weekly_post, confidence=0.95)

print('=== 95% Confidence Interval — Issues per Minggu ===')
print(f"  Sebelum pandas 2.0 : lambda_hat = {ci_pois_pre['theta_hat']:.4f}, "
      f"95% CI = ({ci_pois_pre['lower']:.4f}, {ci_pois_pre['upper']:.4f})")
print(f"  (n minggu = {len(weekly_pre)})")
print()
print(f"  Sesudah pandas 2.0 : lambda_hat = {ci_pois_post['theta_hat']:.4f}, "
      f"95% CI = ({ci_pois_post['lower']:.4f}, {ci_pois_post['upper']:.4f})")
print(f"  (n minggu = {len(weekly_post)})")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

labels = ['Sebelum pandas 2.0', 'Sesudah pandas 2.0']
estimates = [ci_pois_pre['theta_hat'], ci_pois_post['theta_hat']]
errors = [
    [ci_pois_pre['theta_hat'] - ci_pois_pre['lower'], ci_pois_post['theta_hat'] - ci_pois_post['lower']],
    [ci_pois_pre['upper'] - ci_pois_pre['theta_hat'], ci_pois_post['upper'] - ci_pois_post['theta_hat']],
]

ax.errorbar(labels, estimates, yerr=errors, fmt='o', markersize=10,
            capsize=8, color='steelblue', ecolor='steelblue', linewidth=2)
ax.set_title('95% Confidence Interval — Rata-rata Issue per Minggu')
ax.set_ylabel(r'$\hat{\lambda}$ (issues/minggu)')
plt.tight_layout()
plt.savefig('../data/clean/plot_ci_poisson.png', dpi=150)
plt.show()

**Interpretasi:** [TUGAS KAMU — tulis interpretasi di sini]

Panduan (hapus setelah ditulis ulang dengan kalimatmu sendiri):
- Sebutkan kedua interval dan bandingkan lebar masing-masing — interval mana yang lebih lebar, dan mengapa? (perhatikan perbedaan n minggu: pre=108 vs post=37)
- Gunakan bahasa frekuentis yang benar (sama seperti di Bagian 1).
- Apakah kedua interval tumpang tindih (overlap) atau tidak? Sebutkan saja faktanya di sini — pengujian formal soal signifikansi perbedaan akan dilakukan Member D di notebook 04 menggunakan two-sample Z-test.

---
## Ringkasan & Handoff ke Layer Berikutnya

**Ringkasan temuan inferensi:** [TUGAS KAMU — tulis 2-3 poin ringkasan, contoh: nilai-nilai CI yang didapat dan apa artinya secara umum bagi proyek pandas]

**Output untuk layer berikutnya (Member D — Hypothesis Testing):**
- `lambda_hat_pre_v2` dan `lambda_hat_post_v2` beserta CI masing-masing menunjukkan bahwa kedua interval [overlap / tidak overlap — isi sesuai hasil di atas].
- Member D akan menguji secara formal apakah perbedaan rata-rata issue per minggu sebelum dan sesudah pandas 2.0 signifikan secara statistik menggunakan **two-sample Z-test**, serta apakah penurunan ini signifikan dibanding rata-rata historis menggunakan **one-sample Z-test**.